# Smart ALPR IoT — Entrenamiento y Evaluación Real de YOLO

Este cuaderno entrena un detector **Ultralytics YOLO** para localizar placas vehiculares colombianas (clase única `placa`), evalúa su desempeño real sobre datos no vistos (partición `test`), mide la curva de sensibilidad del umbral de confianza, inspecciona hasta **3 casos de falla** y exporta los pesos `best.pt` junto con `sensibilidad_umbral.csv`.

## Requisitos antes de ejecutar

- **Aceleración por hardware en Google Colab:** activa una GPU en **Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU T4**.
- **Duración aproximada:** entre **10 y 25 minutos** en una GPU T4 para un conjunto típico de 500 a 2,000 imágenes con 80 épocas (puede terminar antes por *early stopping* con paciencia de 20 épocas).
- **Flujo recomendado:**
  1. Completa únicamente la celda **3. Configuración**.
  2. Si usas Roboflow, guarda tu clave en los **Secretos de Colab** (icono de llave en la barra izquierda) con el nombre `ROBOFLOW_API_KEY` y habilita el acceso para este cuaderno.
  3. Pulsa **Entorno de ejecución -> Ejecutar todo**.

## 2. Comprobar GPU e instalar Ultralytics

La consola local del repositorio `smart-alpr-iot` carga `models/best.pt` con `ultralytics==8.4.220`. Este entorno instala exactamente la misma versión para garantizar compatibilidad de pesos y arquitectura.

In [ ]:
!nvidia-smi
%pip install -q ultralytics==8.4.220

import torch
import ultralytics

print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No se detectó GPU CUDA. En Google Colab ve a: "
        "Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU T4."
    )

print(f"GPU activa: {torch.cuda.get_device_name(0)}")

## 3. Configuración

Edita **solo esta celda** antes de ejecutar todo el cuaderno:
- Si el dataset está en Google Drive como archivo `.zip`, deja `ORIGEN_DATASET = "drive"` y escribe la ruta en `RUTA_ZIP_DRIVE`.
- Si el dataset está en Roboflow, cambia `ORIGEN_DATASET = "roboflow"`, completa `ROBOFLOW_WORKSPACE`, `ROBOFLOW_PROJECT` y `ROBOFLOW_VERSION`, y guarda tu clave en el secreto `ROBOFLOW_API_KEY` de Colab.

In [ ]:
# --- Origen del dataset ("drive" o "roboflow") ---
ORIGEN_DATASET = "drive"

# Opción A: archivo .zip en Google Drive (requerido si ORIGEN_DATASET == "drive")
RUTA_ZIP_DRIVE = ""

# Opción B: Roboflow (requerido si ORIGEN_DATASET == "roboflow")
ROBOFLOW_WORKSPACE = ""
ROBOFLOW_PROJECT = ""
ROBOFLOW_VERSION = ""
ROBOFLOW_API_KEY = None
if ORIGEN_DATASET == "roboflow":
    from google.colab import userdata

    try:
        ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    except Exception as exc:
        raise ValueError(
            "No se pudo leer el secreto 'ROBOFLOW_API_KEY' desde Google Colab. "
            "Abre el panel Secretos (icono de llave a la izquierda), crea 'ROBOFLOW_API_KEY' "
            "y activa el acceso para este cuaderno."
        ) from exc

# --- Hiperparámetros de entrenamiento ---
MODELO_BASE = "yolo11n.pt"
EPOCHS = 80
IMGSZ = 640
BATCH = 16
PATIENCE = 20
SEED = 42

# --- Rutas locales en Colab y carpeta de respaldo en Drive ---
DIRECTORIO_DATASET = "/content/dataset"
PROYECTO_RUNS = "/content/runs_alpr"
NOMBRE_EXPERIMENTO = "yolo11n_placas"
CARPETA_EXPORT_DRIVE = "/content/drive/MyDrive/smart_alpr_iot"

## 4. Obtener el dataset

Carga el conjunto de datos etiquetado en formato YOLO desde Google Drive (`.zip`) o desde Roboflow (`yolov8`). Si falta algún parámetro obligatorio en la configuración, la celda se detiene con instrucciones claras.

In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

origen = ORIGEN_DATASET.strip().lower()
if origen not in {"drive", "roboflow"}:
    raise ValueError(
        f"ORIGEN_DATASET debe ser 'drive' o 'roboflow', pero se recibió: {ORIGEN_DATASET!r}. "
        "Corrige la celda 3 (Configuración)."
    )

dataset_root = Path(DIRECTORIO_DATASET)
if dataset_root.exists():
    shutil.rmtree(dataset_root)
dataset_root.mkdir(parents=True, exist_ok=True)

if origen == "drive":
    if not RUTA_ZIP_DRIVE or not RUTA_ZIP_DRIVE.strip():
        raise ValueError(
            "Falta RUTA_ZIP_DRIVE en la celda de configuración. "
            "Monta o sube tu archivo .zip a Google Drive e indica la ruta completa "
            "(por ejemplo: /content/drive/MyDrive/carpeta/dataset_yolo.zip)."
        )

    from google.colab import drive

    drive.mount("/content/drive")
    zip_path = Path(RUTA_ZIP_DRIVE.strip())
    if not zip_path.is_file():
        raise ValueError(
            f"No se encontró el archivo .zip en Google Drive: {zip_path}. "
            "Verifica la ruta exacta en RUTA_ZIP_DRIVE."
        )
    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f"El archivo indicado no es un .zip válido: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dataset_root)

    candidatos_yaml = [
        p for p in sorted(dataset_root.rglob("data.yaml")) if "__MACOSX" not in p.parts
    ]
    if not candidatos_yaml:
        candidatos_yaml = [
            p
            for ext in ("*.yaml", "*.yml")
            for p in sorted(dataset_root.rglob(ext))
            if "__MACOSX" not in p.parts
        ]
    if not candidatos_yaml:
        raise FileNotFoundError(
            f"Se descomprimió {zip_path}, pero no se encontró ningún archivo data.yaml en {dataset_root}."
        )
    candidatos_yaml.sort(key=lambda p: (len(p.parts), str(p)))
    data_yaml_path = candidatos_yaml[0]

else:
    faltantes = []
    if not ROBOFLOW_WORKSPACE or not ROBOFLOW_WORKSPACE.strip():
        faltantes.append("ROBOFLOW_WORKSPACE")
    if not ROBOFLOW_PROJECT or not ROBOFLOW_PROJECT.strip():
        faltantes.append("ROBOFLOW_PROJECT")
    if not ROBOFLOW_VERSION or not str(ROBOFLOW_VERSION).strip():
        faltantes.append("ROBOFLOW_VERSION")
    if not ROBOFLOW_API_KEY or not str(ROBOFLOW_API_KEY).strip():
        faltantes.append("ROBOFLOW_API_KEY (secreto de Colab)")

    if faltantes:
        raise ValueError(
            "Falta completar la configuración de Roboflow: "
            + ", ".join(faltantes)
            + ". Llena los campos en la celda 3 y configura el secreto ROBOFLOW_API_KEY en Colab."
        )

    try:
        version_num = int(str(ROBOFLOW_VERSION).strip())
    except ValueError as exc:
        raise ValueError(
            f"ROBOFLOW_VERSION debe ser un número entero (ej. '1'), se recibió: {ROBOFLOW_VERSION!r}"
        ) from exc

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "roboflow"])
    from roboflow import Roboflow

    rf = Roboflow(api_key=str(ROBOFLOW_API_KEY).strip())
    rf_project = rf.workspace(ROBOFLOW_WORKSPACE.strip()).project(ROBOFLOW_PROJECT.strip())
    rf_dataset = rf_project.version(version_num).download("yolov8", location=str(dataset_root))
    data_yaml_path = Path(rf_dataset.location) / "data.yaml"
    if not data_yaml_path.is_file():
        raise FileNotFoundError(f"No se encontró data.yaml tras descargar en {rf_dataset.location}")

print(f"Dataset listo. Archivo de configuración: {data_yaml_path}")

## 5. Revisar el dataset

Se verifica que `data.yaml` defina las tres particiones obligatorias (`train`, `val` y `test`), se contabilizan las imágenes y anotaciones reales por partición y se visualizan 6 imágenes de entrenamiento con sus cajas etiquetadas.

In [ ]:
import random
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import yaml
from PIL import Image, ImageOps
from ultralytics.data.utils import IMG_FORMATS, img2label_paths

with open(data_yaml_path, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

if not isinstance(data_cfg, dict):
    raise ValueError(f"El archivo {data_yaml_path} no tiene una estructura YAML válida.")

raw_names = data_cfg.get("names")
if isinstance(raw_names, list):
    nombres_clases = {i: str(n) for i, n in enumerate(raw_names)}
elif isinstance(raw_names, dict):
    nombres_clases = {int(k): str(v) for k, v in raw_names.items()}
else:
    raise ValueError("data.yaml debe incluir el campo 'names' con el nombre de las clases (ej. ['placa']).")

INSTRUCCIONES_TEST = (
    "El proyecto exige evaluar sobre una partición 'test' de datos no vistos.\n"
    "Cómo crearla:\n"
    "- En Roboflow: genera una nueva versión del dataset con reparto 70% train / 20% valid / 10% test.\n"
    "- En archivo .zip de Drive: separa aprox. 10% de tus fotos originales y sus archivos .txt "
    "en las carpetas test/images y test/labels (o images/test y labels/test) y agrega la línea "
    "'test: test/images' (o la ruta correspondiente) dentro de data.yaml."
)

for particion in ("train", "val", "test"):
    if not data_cfg.get(particion):
        if particion == "test":
            raise ValueError(f"Falta la partición 'test' en {data_yaml_path}.\n{INSTRUCCIONES_TEST}")
        raise ValueError(f"Falta la partición obligatoria '{particion}' en {data_yaml_path}.")

yaml_dir = data_yaml_path.parent.resolve()
raw_root = data_cfg.get("path")
if raw_root:
    candidato_root = Path(str(raw_root))
    if not candidato_root.is_absolute():
        candidato_root = (yaml_dir / candidato_root).resolve()
    if candidato_root.exists():
        base_dir = candidato_root
    else:
        print(
            f"Aviso: 'path: {raw_root}' en data.yaml no existe en Colab; "
            f"se ajusta automáticamente a {yaml_dir}."
        )
        base_dir = yaml_dir
        data_cfg["path"] = str(yaml_dir)
        with open(data_yaml_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)
else:
    base_dir = yaml_dir


def resolver_rutas_split(raiz: Path, respaldo_dir: Path, entrada: str | list[str]) -> list[Path]:
    """Resuelve una o varias rutas de partición YOLO igual que Ultralytics."""
    items = entrada if isinstance(entrada, list) else [entrada]
    resueltas: list[Path] = []
    for item in items:
        texto = str(item)
        p = Path(texto)
        if p.is_absolute() and p.exists():
            resueltas.append(p.resolve())
            continue
        candidato = (raiz / texto).resolve()
        if not candidato.exists() and texto.startswith("../"):
            candidato = (raiz / texto[3:]).resolve()
        if not candidato.exists():
            candidato = (respaldo_dir / texto).resolve()
            if not candidato.exists() and texto.startswith("../"):
                candidato = (respaldo_dir / texto[3:]).resolve()
        resueltas.append(candidato)
    return resueltas


def listar_imagenes_split(rutas: list[Path]) -> list[Path]:
    """Lista las imágenes válidas de una partición desde carpetas, archivos o listas .txt."""
    encontradas: list[Path] = []
    for p in rutas:
        if p.is_dir():
            for archivo in p.rglob("*"):
                if archivo.is_file() and archivo.suffix.lower().lstrip(".") in IMG_FORMATS:
                    encontradas.append(archivo)
        elif p.is_file() and p.suffix.lower().lstrip(".") in IMG_FORMATS:
            encontradas.append(p)
        elif p.is_file():
            with open(p, "r", encoding="utf-8") as f:
                for linea in f.read().splitlines():
                    linea = linea.strip()
                    if not linea:
                        continue
                    img_p = (p.parent / linea[2:]) if linea.startswith("./") else Path(linea)
                    if img_p.suffix.lower().lstrip(".") in IMG_FORMATS:
                        encontradas.append(img_p)
    return sorted(set(encontradas))


imagenes_por_split: dict[str, list[Path]] = {}
etiquetas_por_split: dict[str, list[Path]] = {}

print(f"Clases detectadas en data.yaml: {nombres_clases}")
print("\n=== Resumen por partición ===")

for particion in ("train", "val", "test"):
    rutas = resolver_rutas_split(base_dir, yaml_dir, data_cfg[particion])
    faltan_rutas = [str(r) for r in rutas if not r.exists()]
    if faltan_rutas:
        detalle = ", ".join(faltan_rutas)
        if particion == "test":
            raise ValueError(f"No existe la carpeta/ruta de la partición 'test': {detalle}\n{INSTRUCCIONES_TEST}")
        raise ValueError(f"No existe la carpeta/ruta de la partición '{particion}': {detalle}")

    imgs = listar_imagenes_split(rutas)
    if not imgs:
        if particion == "test":
            raise ValueError(f"La partición 'test' no contiene imágenes.\n{INSTRUCCIONES_TEST}")
        raise ValueError(f"La partición '{particion}' no contiene imágenes válidas.")

    labels = [Path(lp) for lp in img2label_paths([str(p) for p in imgs])]
    archivos_txt = 0
    imgs_con_cajas = 0
    total_cajas = 0

    for lp in labels:
        if lp.is_file():
            archivos_txt += 1
            with open(lp, "r", encoding="utf-8") as f:
                lineas = [ln.strip() for ln in f.read().splitlines() if ln.strip()]
            if lineas:
                imgs_con_cajas += 1
                total_cajas += len(lineas)

    imagenes_por_split[particion] = imgs
    etiquetas_por_split[particion] = labels
    print(
        f"- {particion:5s}: {len(imgs):4d} imágenes | "
        f"{archivos_txt:4d} archivos .txt | "
        f"{imgs_con_cajas:4d} con etiquetas | "
        f"{total_cajas:4d} cajas totales"
    )

# Visualizar 6 imágenes de entrenamiento con sus cajas reales
pares_train = list(zip(imagenes_por_split["train"], etiquetas_por_split["train"]))
pares_con_etiqueta = [
    (im, lb) for im, lb in pares_train if lb.is_file() and lb.read_text(encoding="utf-8").strip()
]
pool_muestra = pares_con_etiqueta if len(pares_con_etiqueta) >= 6 else pares_train
rng = random.Random(SEED)
muestra_train = rng.sample(pool_muestra, k=min(6, len(pool_muestra)))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.ravel()

for ax, (img_path, lb_path) in zip(axes_flat, muestra_train):
    with Image.open(img_path) as im_pil:
        im_pil = ImageOps.exif_transpose(im_pil).convert("RGB")
        w_img, h_img = im_pil.size
        ax.imshow(im_pil)

    n_cajas = 0
    if lb_path.is_file():
        for linea in lb_path.read_text(encoding="utf-8").splitlines():
            partes = linea.strip().split()
            if len(partes) == 5:
                cls_id = int(float(partes[0]))
                xc, yc, bw, bh = map(float, partes[1:5])
                x1 = (xc - bw / 2.0) * w_img
                y1 = (yc - bh / 2.0) * h_img
                rect_w = bw * w_img
                rect_h = bh * h_img
            elif len(partes) >= 7 and (len(partes) - 1) % 2 == 0:
                cls_id = int(float(partes[0]))
                coords = list(map(float, partes[1:]))
                xs = [coords[i] * w_img for i in range(0, len(coords), 2)]
                ys = [coords[i] * h_img for i in range(1, len(coords), 2)]
                x1, y1 = min(xs), min(ys)
                rect_w, rect_h = max(xs) - x1, max(ys) - y1
            else:
                continue

            n_cajas += 1
            rect = patches.Rectangle((x1, y1), rect_w, rect_h, linewidth=2, edgecolor="#10B981", facecolor="none")
            ax.add_patch(rect)
            etiqueta = nombres_clases.get(cls_id, str(cls_id))
            ax.text(
                x1,
                max(0.0, y1 - 4.0),
                etiqueta,
                color="black",
                fontsize=9,
                bbox=dict(facecolor="#10B981", edgecolor="none", pad=1.5),
            )

    ax.set_title(f"{img_path.name} ({n_cajas} cajas)", fontsize=10)
    ax.axis("off")

for ax in axes_flat[len(muestra_train):]:
    ax.axis("off")

fig.suptitle("Muestra de entrenamiento con etiquetas reales", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Entrenamiento del modelo YOLO

Entrena `yolo11n.pt` mediante aprendizaje por transferencia con los hiperparámetros definidos en la celda 3 (`EPOCHS=80`, `IMGSZ=640`, `BATCH=16`, `PATIENCE=20`, `SEED=42`). El mejor punto de control según `val` queda guardado como `best.pt`.

In [ ]:
from ultralytics import YOLO

device_entrenamiento = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(MODELO_BASE)

resultados_entrenamiento = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    device=device_entrenamiento,
    project=PROYECTO_RUNS,
    name=NOMBRE_EXPERIMENTO,
    exist_ok=True,
    plots=True,
    save=True,
)

best_weights_path = Path(model.trainer.best)
if not best_weights_path.is_file():
    raise FileNotFoundError(
        f"El entrenamiento finalizó pero no se encontró best.pt en {best_weights_path}"
    )

print(f"Carpeta del experimento: {model.trainer.save_dir}")
print(f"Mejores pesos guardados en: {best_weights_path}")

## 7. Evaluación en la partición de prueba (datos no vistos)

Se evalúa `best.pt` exclusivamente sobre la partición `test` (`split="test"`), reportando exactamente las métricas de Ultralytics (`Precision`, `Recall`, `mAP50`, `mAP50-95`) y mostrando las gráficas generadas.

In [ ]:
import matplotlib.image as mpimg

best_model = YOLO(str(best_weights_path))
metrics = best_model.val(
    data=str(data_yaml_path),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=device_entrenamiento,
    plots=True,
    project=PROYECTO_RUNS,
    name=f"{NOMBRE_EXPERIMENTO}_eval_test",
    exist_ok=True,
)

print("=== Métricas medidas en partición test (datos no vistos) ===")
print(f"Precision (metrics.box.mp):  {metrics.box.mp:.4f}")
print(f"Recall (metrics.box.mr):     {metrics.box.mr:.4f}")
print(f"mAP50 (metrics.box.map50):   {metrics.box.map50:.4f}")
print(f"mAP50-95 (metrics.box.map):  {metrics.box.map:.4f}")

print("\nDetalle devuelto por Ultralytics (metrics.results_dict):")
for clave, valor in metrics.results_dict.items():
    print(f"  {clave}: {valor:.4f}")

val_dir = Path(metrics.save_dir)
graficas_preferidas = [
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
]
rutas_graficas = [val_dir / nombre for nombre in graficas_preferidas if (val_dir / nombre).is_file()]
if not rutas_graficas:
    rutas_graficas = sorted(val_dir.glob("*.png"))

if not rutas_graficas:
    print(f"No se encontraron gráficas PNG en {val_dir}.")
else:
    n_cols = 2
    n_rows = (len(rutas_graficas) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
    axes_list = list(axes.ravel()) if hasattr(axes, "ravel") else [axes]
    for ax, ruta in zip(axes_list, rutas_graficas):
        ax.imshow(mpimg.imread(ruta))
        ax.set_title(ruta.name, fontsize=11)
        ax.axis("off")
    for ax in axes_list[len(rutas_graficas):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 8. Sensibilidad del umbral de confianza (medida)

A partir de las curvas medidas por Ultralytics en la validación sobre `test` (`metrics.box.px`, `metrics.box.p_curve`, `metrics.box.r_curve` y `metrics.box.f1_curve`), se extrae la precisión, el *recall* y el F1-score reales en los umbrales `0.25`, `0.35`, `0.45`, `0.55`, `0.65` y `0.75`, y se guarda el resultado en `sensibilidad_umbral.csv`.

In [ ]:
import numpy as np
import pandas as pd

px = np.asarray(metrics.box.px, dtype=float)
p_curve_raw = np.asarray(metrics.box.p_curve, dtype=float)
r_curve_raw = np.asarray(metrics.box.r_curve, dtype=float)
f1_curve_raw = np.asarray(metrics.box.f1_curve, dtype=float)

if px.size == 0 or p_curve_raw.size == 0 or r_curve_raw.size == 0 or f1_curve_raw.size == 0:
    raise RuntimeError(
        "Ultralytics no generó curvas P/R por confianza en la validación de 'test'. "
        "Comprueba que la partición 'test' contenga etiquetas válidas."
    )

p_curve = p_curve_raw.mean(axis=0)
r_curve = r_curve_raw.mean(axis=0)
f1_curve = f1_curve_raw.mean(axis=0)

umbrales_evaluados = [0.25, 0.35, 0.45, 0.55, 0.65, 0.75]
filas_sensibilidad = []
for umbral in umbrales_evaluados:
    prec = float(np.interp(umbral, px, p_curve))
    rec = float(np.interp(umbral, px, r_curve))
    f1_val = (2.0 * prec * rec / (prec + rec)) if (prec + rec) > 0.0 else 0.0
    filas_sensibilidad.append(
        {
            "umbral": umbral,
            "precision": round(prec, 4),
            "recall": round(rec, 4),
            "f1": round(f1_val, 4),
        }
    )

df_sensibilidad = pd.DataFrame(filas_sensibilidad)
idx_mejor_tabla = int(df_sensibilidad["f1"].idxmax())
mejor_umbral_tabla = float(df_sensibilidad.loc[idx_mejor_tabla, "umbral"])
mejor_f1_tabla = float(df_sensibilidad.loc[idx_mejor_tabla, "f1"])

idx_mejor_curva = int(np.argmax(f1_curve))
mejor_umbral_curva = float(px[idx_mejor_curva])
mejor_f1_curva = float(f1_curve[idx_mejor_curva])

df_sensibilidad["mejor_en_tabla"] = df_sensibilidad["umbral"] == mejor_umbral_tabla
ruta_csv_sensibilidad = Path("sensibilidad_umbral.csv")
df_sensibilidad.to_csv(ruta_csv_sensibilidad, index=False)

print("=== Sensibilidad del umbral de confianza (medida en test) ===")
print(df_sensibilidad.to_string(index=False))
print(
    f"\nUmbral con mejor F1 entre los 6 evaluados: {mejor_umbral_tabla:.2f} "
    f"(F1 = {mejor_f1_tabla:.4f})"
)
print(
    f"Umbral con mejor F1 en la curva completa (1000 puntos de px/f1_curve): "
    f"{mejor_umbral_curva:.3f} (F1 = {mejor_f1_curva:.4f})"
)
print(f"Tabla guardada en: {ruta_csv_sensibilidad.resolve()}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(px, p_curve, label="Precisión", color="#10B981", linewidth=2)
ax.plot(px, r_curve, label="Recall", color="#3B82F6", linewidth=2)
ax.plot(px, f1_curve, label="F1", color="#F59E0B", linewidth=2)
ax.scatter(df_sensibilidad["umbral"], df_sensibilidad["f1"], color="#EF4444", zorder=5, label="F1 en umbrales")
ax.axvline(mejor_umbral_tabla, color="#EF4444", linestyle="--", linewidth=1.5, label=f"Mejor tabla ({mejor_umbral_tabla:.2f})")
ax.set_title("Sensibilidad medida del umbral de confianza (partición test)")
ax.set_xlabel("Umbral de confianza")
ax.set_ylabel("Métrica")
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 9. Tres casos de falla en la partición de prueba

Se ejecuta inferencia sobre las imágenes de `test` con el mejor umbral medido en la tabla (`mejor_umbral_tabla`), se emparejan las predicciones con las cajas reales usando **IoU >= 0.5** y se muestran hasta **3 imágenes** que presenten falsos negativos (placas no detectadas) o falsos positivos (detecciones erróneas).
- **Verde:** caja real etiquetada.
- **Rojo:** caja predicha por el modelo.

In [ ]:
import cv2

CONF_FALLOS = mejor_umbral_tabla


def calcular_iou_matriz(cajas_a: np.ndarray, cajas_b: np.ndarray) -> np.ndarray:
    """Calcula la matriz IoU (N x M) entre dos conjuntos de cajas xyxy."""
    x1 = np.maximum(cajas_a[:, None, 0], cajas_b[None, :, 0])
    y1 = np.maximum(cajas_a[:, None, 1], cajas_b[None, :, 1])
    x2 = np.minimum(cajas_a[:, None, 2], cajas_b[None, :, 2])
    y2 = np.minimum(cajas_a[:, None, 3], cajas_b[None, :, 3])

    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = np.maximum(0.0, cajas_a[:, 2] - cajas_a[:, 0]) * np.maximum(0.0, cajas_a[:, 3] - cajas_a[:, 1])
    area_b = np.maximum(0.0, cajas_b[:, 2] - cajas_b[:, 0]) * np.maximum(0.0, cajas_b[:, 3] - cajas_b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0.0, inter / union, 0.0)


def cargar_cajas_reales(label_path: Path, ancho: int, alto: int) -> tuple[np.ndarray, np.ndarray]:
    """Lee etiquetas YOLO de un archivo .txt y devuelve (cajas_xyxy, clases)."""
    if not label_path.is_file():
        return np.zeros((0, 4), dtype=float), np.zeros((0,), dtype=int)

    cajas: list[list[float]] = []
    clases: list[int] = []
    for linea in label_path.read_text(encoding="utf-8").splitlines():
        partes = linea.strip().split()
        if len(partes) == 5:
            cls_id = int(float(partes[0]))
            xc, yc, bw, bh = map(float, partes[1:5])
            cajas.append([
                (xc - bw / 2.0) * ancho,
                (yc - bh / 2.0) * alto,
                (xc + bw / 2.0) * ancho,
                (yc + bh / 2.0) * alto,
            ])
            clases.append(cls_id)
        elif len(partes) >= 7 and (len(partes) - 1) % 2 == 0:
            cls_id = int(float(partes[0]))
            coords = list(map(float, partes[1:]))
            xs = [coords[i] * ancho for i in range(0, len(coords), 2)]
            ys = [coords[i] * alto for i in range(1, len(coords), 2)]
            cajas.append([min(xs), min(ys), max(xs), max(ys)])
            clases.append(cls_id)

    if not cajas:
        return np.zeros((0, 4), dtype=float), np.zeros((0,), dtype=int)
    return np.asarray(cajas, dtype=float), np.asarray(clases, dtype=int)


rutas_test = [str(p) for p in imagenes_por_split["test"]]
predicciones = best_model.predict(
    source=rutas_test,
    imgsz=IMGSZ,
    conf=CONF_FALLOS,
    iou=0.5,
    device=device_entrenamiento,
    verbose=False,
    stream=True,
)

casos_falla = []
total_fn = 0
total_fp = 0
total_tp = 0

for res in predicciones:
    alto, ancho = res.orig_shape
    ruta_img = Path(res.path)
    ruta_lbl = Path(img2label_paths([res.path])[0])
    gt_boxes, gt_cls = cargar_cajas_reales(ruta_lbl, ancho=ancho, alto=alto)

    if res.boxes is not None and len(res.boxes) > 0:
        pred_boxes = res.boxes.xyxy.cpu().numpy()
        pred_confs = res.boxes.conf.cpu().numpy()
        pred_cls = res.boxes.cls.cpu().numpy().astype(int)
    else:
        pred_boxes = np.zeros((0, 4), dtype=float)
        pred_confs = np.zeros((0,), dtype=float)
        pred_cls = np.zeros((0,), dtype=int)

    gt_emparejado = np.zeros(len(gt_boxes), dtype=bool)
    pred_emparejado = np.zeros(len(pred_boxes), dtype=bool)
    tp_img = 0

    if len(gt_boxes) > 0 and len(pred_boxes) > 0:
        iou_mat = calcular_iou_matriz(gt_boxes, pred_boxes)
        orden_pred = np.argsort(-pred_confs, kind="stable")
        for di in orden_pred:
            candidatos = [
                gi
                for gi in range(len(gt_boxes))
                if not gt_emparejado[gi] and gt_cls[gi] == pred_cls[di] and iou_mat[gi, di] >= 0.5
            ]
            if candidatos:
                mejor_gi = max(candidatos, key=lambda gi: iou_mat[gi, di])
                gt_emparejado[mejor_gi] = True
                pred_emparejado[di] = True
                tp_img += 1

    fn_img = int((~gt_emparejado).sum())
    fp_img = int((~pred_emparejado).sum())
    total_tp += tp_img
    total_fn += fn_img
    total_fp += fp_img

    if fn_img > 0 or fp_img > 0:
        img_orig = res.orig_img
        img_rgb = (
            cv2.cvtColor(img_orig, cv2.COLOR_GRAY2RGB)
            if img_orig.ndim == 2
            else cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)
        )
        casos_falla.append(
            {
                "ruta": ruta_img,
                "img_rgb": img_rgb,
                "gt_boxes": gt_boxes,
                "pred_boxes": pred_boxes,
                "pred_confs": pred_confs,
                "tp": tp_img,
                "fn": fn_img,
                "fp": fp_img,
            }
        )

print(f"Evaluación de fallas en {len(rutas_test)} imágenes de 'test' (conf >= {CONF_FALLOS:.2f}, IoU >= 0.5):")
print(f"  TP totales: {total_tp} | FN totales: {total_fn} | FP totales: {total_fp}")
print(f"  Imágenes con al menos un error (FN o FP): {len(casos_falla)}")

casos_falla.sort(key=lambda c: (-(c["fn"] + c["fp"]), c["ruta"].name))
casos_mostrar = casos_falla[:3]

if not casos_mostrar:
    print("No se encontraron falsos negativos ni falsos positivos en 'test' con este umbral.")
else:
    fig, axes = plt.subplots(1, len(casos_mostrar), figsize=(6 * len(casos_mostrar), 6))
    axes_list = list(axes) if len(casos_mostrar) > 1 else [axes]

    for idx, (ax, caso) in enumerate(zip(axes_list, casos_mostrar), start=1):
        ax.imshow(caso["img_rgb"])
        for x1, y1, x2, y2 in caso["gt_boxes"]:
            rect_gt = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2.5,
                edgecolor="#10B981",
                facecolor="none",
                label="Real (GT)",
            )
            ax.add_patch(rect_gt)

        for (x1, y1, x2, y2), conf in zip(caso["pred_boxes"], caso["pred_confs"]):
            rect_pred = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2.5,
                linestyle="--",
                edgecolor="#EF4444",
                facecolor="none",
                label="Predicha",
            )
            ax.add_patch(rect_pred)
            ax.text(
                x1,
                max(0.0, y1 - 4.0),
                f"{conf:.2f}",
                color="white",
                fontsize=9,
                bbox=dict(facecolor="#EF4444", edgecolor="none", pad=1.5),
            )

        handles, labels = ax.get_legend_handles_labels()
        leyenda_unica = dict(zip(labels, handles))
        if leyenda_unica:
            ax.legend(leyenda_unica.values(), leyenda_unica.keys(), loc="lower right", fontsize=9)

        ax.set_title(
            f"Caso {idx}: {caso['ruta'].name}\nTP={caso['tp']} | FN={caso['fn']} | FP={caso['fp']}",
            fontsize=10,
        )
        ax.axis("off")
        print(
            f"- Caso {idx}: {caso['ruta'].name} -> "
            f"TP={caso['tp']}, FN={caso['fn']}, FP={caso['fp']}"
        )

    plt.tight_layout()
    plt.show()

### Análisis humano de los 3 casos de falla

Completa esta celda después de ejecutar el cuaderno con las observaciones reales de las imágenes mostradas arriba:

1. **Caso 1**  
   - **Archivo:** *(completar tras ejecutar)*  
   - **Tipo de falla (FN / FP):** *(completar)*  
   - **Por qué falló (ángulo, reflejo, desenfoque, placa pequeña, oclusión, elemento confundido):** *(escribir explicación)*  
   - **Mitigación propuesta:** *(escribir posible mejora)*  

2. **Caso 2**  
   - **Archivo:** *(completar tras ejecutar)*  
   - **Tipo de falla (FN / FP):** *(completar)*  
   - **Por qué falló:** *(escribir explicación)*  
   - **Mitigación propuesta:** *(escribir posible mejora)*  

3. **Caso 3**  
   - **Archivo:** *(completar tras ejecutar)*  
   - **Tipo de falla (FN / FP):** *(completar)*  
   - **Por qué falló:** *(escribir explicación)*  
   - **Mitigación propuesta:** *(escribir posible mejora)*

## 10. Exportar pesos y sensibilidad a Drive y descarga directa

Guarda `best.pt` y `sensibilidad_umbral.csv` en Google Drive y dispara la descarga directa al equipo local.

In [ ]:
from google.colab import drive, files

ruta_sensibilidad = Path("sensibilidad_umbral.csv")
if not best_weights_path.is_file():
    raise FileNotFoundError(f"No se encontró el modelo entrenado en {best_weights_path}")
if not ruta_sensibilidad.is_file():
    raise FileNotFoundError(f"No se encontró el archivo {ruta_sensibilidad}")

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

destino_drive = Path(CARPETA_EXPORT_DRIVE)
destino_drive.mkdir(parents=True, exist_ok=True)

destino_best = destino_drive / "best.pt"
destino_csv = destino_drive / "sensibilidad_umbral.csv"
shutil.copy2(best_weights_path, destino_best)
shutil.copy2(ruta_sensibilidad, destino_csv)

print(f"Pesos copiados en Drive: {destino_best}")
print(f"Sensibilidad copiada en Drive: {destino_csv}")

files.download(str(best_weights_path))
files.download(str(ruta_sensibilidad))

print("\nNota: si el navegador bloquea la segunda descarga, permite descargas múltiples o recupéralos desde Drive.")
print("=== Instrucción final en tu repositorio local ===")
print("1. Copia el archivo descargado a: models/best.pt")
print("2. Ejecuta: .venv/bin/python -m src.benchmark_ab --json medicion.json")

### Siguiente paso en el repositorio local `smart-alpr-iot`

1. Coloca los pesos descargados en `models/best.pt` dentro del repositorio.
2. Ejecuta desde la raíz del proyecto para medir el pipeline completo (YOLO + preprocesamiento + EasyOCR) sobre las 11 fotos reales de `data/samples/placas_de_prueba/`:

```bash
.venv/bin/python -m src.benchmark_ab --json medicion.json
```

## 11. Si el dataset es propio: cómo etiquetar y exportar

Si vas a construir tu propio conjunto de placas vehiculares colombianas:

1. **Captura de imágenes:** reúne fotos variadas (carros y motos, día y noche, distintas distancias, ángulos y condiciones de iluminación).
2. **Herramienta de etiquetado:** sube las imágenes a **Roboflow** o **CVAT**.
3. **Clase única:** crea una sola clase llamada **`placa`** (índice `0`) y ajusta la caja delimitadora exactamente alrededor de la matrícula (sin incluir el parachoques completo).
4. **Particiones 70 / 20 / 10:** divide las fotos **originales** antes de generar aumentación de datos:
   - **70% `train`** (entrenamiento)
   - **20% `val`** (validación durante el entrenamiento)
   - **10% `test`** (prueba final sobre datos nunca vistos; no mezclar copias aumentadas de `train` en `test`)
5. **Exportación en formato YOLO:**
   - En Roboflow: exporta como **YOLOv8** (puede descargarse directo en este cuaderno con `ORIGEN_DATASET = "roboflow"` o como `.zip` para Drive).
   - En CVAT: exporta como **YOLO 1.1** o **Ultralytics YOLO Detection 1.0**, organiza las carpetas `train`, `val` y `test` con sus subcarpetas `images/` y `labels/`, incluye `data.yaml` con `names: ['placa']`, comprímelo en `.zip`, súbelo a Google Drive y configura `ORIGEN_DATASET = "drive"` en la celda 3.